# Wire Magnetic Field Simulation for NV Sensing Comparison

This notebook simulates magnetic fields from current-carrying wires to compare with experimental NV-center magnetometry data.

## Overview
- Load experimental B-field data from B_results.csv
- Define wire geometry and current parameters
- Implement Biot-Savart law for magnetic field calculation
- Generate simulated B-field maps on the same grid
- Compare simulated vs. measured fields
- Include parameter fitting capabilities

## Physics
The magnetic field from a current-carrying wire is calculated using the Biot-Savart law:

\[\mathbf{B}(\mathbf{r}) = \frac{\mu_0 I}{4\pi} \int \frac{d\mathbf{l} \times (\mathbf{r} - \mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|^3}\]

where $I$ is current, $d\mathbf{l}$ is a line element, and $\mathbf{r}'$ is the position along the wire.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import constants
import os

# Constants
MU_0 = constants.mu_0  # Vacuum permeability [T·m/A] = [H/m]

# File paths
B_RESULTS_CSV = "../B_results.csv"
SIMULATION_RESULTS_CSV = "simulation_results.csv"

# Plot settings
CMAP_DIVERGING = "seismic"
CMAP_MAGNITUDE = "viridis"
plt.rcParams["figure.figsize"] = (10, 8)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True

## 1. Load Experimental B-field Data

Load the reconstructed B-field data from the NV sensing experiment.

In [ ]:
# Load experimental data
if not os.path.exists(B_RESULTS_CSV):
    raise FileNotFoundError(f"Experimental data file not found: {B_RESULTS_CSV}")

exp_df = pd.read_csv(B_RESULTS_CSV)

# Extract metadata
metadata_df = exp_df[exp_df['record_type'] == 'metadata']
grid_size = int(metadata_df[metadata_df['value_nT'] == 'grid_size']['unit'].iloc[0])
extent_um = eval(metadata_df[metadata_df['value_nT'] == 'extent_um']['unit'].iloc[0])
off_time = eval(metadata_df[metadata_df['value_nT'] == 'off_time']['unit'].iloc[0])
neg_time = float(metadata_df[metadata_df['value_nT'] == 'neg_time']['unit'].iloc[0])
pos_time = float(metadata_df[metadata_df['value_nT'] == 'pos_time']['unit'].iloc[0])

# Extract B-field data
b_df = exp_df[exp_df['record_type'] == 'B']

print("Loaded experimental data:")
print(f"Grid size: {grid_size}x{grid_size}")
print(f"Extent: {extent_um} µm")
print(f"Total B-field records: {len(b_df)}")
print(f"Phases: {sorted(b_df['phase'].unique())}")
print(f"Components: {sorted(b_df['component'].unique())}")

# Display metadata
display(metadata_df[['value_nT', 'unit']])

## 2. Define Wire Geometry and Current Parameters

Set up the simulation parameters for the current-carrying wire.

In [ ]:
# Wire geometry parameters
wire_params = {
    'center_x_um': 0.0,      # Wire center x position [µm]
    'center_y_um': 0.0,      # Wire center y position [µm] 
    'length_um': 200.0,      # Wire length along y-axis [µm]
    'width_um': 50.0,        # Wire width (for finite-width model) [µm]
    'height_um': 10.0,       # Wire height above xy-plane [µm]
    'orientation': 'y',      # Wire orientation: 'x' or 'y'
}

# Current parameters
current_params = {
    'I_pos_mA': 2.0,         # Positive current [mA]
    'I_neg_mA': -2.0,        # Negative current [mA]
}

# Simulation grid (match experimental)
x_um = np.linspace(extent_um[0], extent_um[1], grid_size)
y_um = np.linspace(extent_um[2], extent_um[3], grid_size)
X_um, Y_um = np.meshgrid(x_um, y_um)

# Convert to meters for calculations
X_m = X_um * 1e-6
Y_m = Y_um * 1e-6
Z_m = np.zeros_like(X_m)  # Observation plane at z=0

print("Wire parameters:")
for key, value in wire_params.items():
    print(f"  {key}: {value}")

print("\nCurrent parameters:")
for key, value in current_params.items():
    print(f"  {key}: {value}")

print(f"\nSimulation grid: {grid_size}x{grid_size} points")
print(f"Extent: {extent_um} µm")

## 3. Biot-Savart Law Implementation

Implement the magnetic field calculation using the Biot-Savart law.

In [ ]:
def biot_savart_wire(X, Y, Z, wire_params, I_A):
    """
    Calculate magnetic field from a finite-width wire using Biot-Savart law.
    
    Parameters:
    X, Y, Z: 2D arrays of observation points [m]
    wire_params: dict with wire geometry
    I_A: current [A]
    
    Returns:
    Bx, By, Bz: magnetic field components [T]
    """
    
    # Convert wire parameters to meters
    cx = wire_params['center_x_um'] * 1e-6
    cy = wire_params['center_y_um'] * 1e-6
    length = wire_params['length_um'] * 1e-6
    width = wire_params['width_um'] * 1e-6
    height = wire_params['height_um'] * 1e-6
    
    # Wire orientation
    if wire_params['orientation'] == 'y':
        # Wire along y-axis
        y_start = cy - length/2
        y_end = cy + length/2
        x_start = cx - width/2
        x_end = cx + width/2
    else:
        # Wire along x-axis
        x_start = cx - length/2
        x_end = cx + length/2
        y_start = cy - width/2
        y_end = cy + width/2
    
    # Number of filaments for discretization
    n_filaments_length = 20
    n_filaments_width = 10
    
    # Initialize B field
    Bx = np.zeros_like(X)
    By = np.zeros_like(X)
    Bz = np.zeros_like(X)
    
    # Loop over wire cross-section (filaments)
    for ix in range(n_filaments_width):
        for iy in range(n_filaments_length):
            # Position of current filament
            if wire_params['orientation'] == 'y':
                x_fil = x_start + (ix + 0.5) * width / n_filaments_width
                y_fil = y_start + (iy + 0.5) * length / n_filaments_length
            else:
                x_fil = x_start + (ix + 0.5) * length / n_filaments_length
                y_fil = y_start + (iy + 0.5) * width / n_filaments_width
            
            z_fil = height
            
            # Current density (uniform)
            dl = np.array([0, 0, 0])  # Initialize
            if wire_params['orientation'] == 'y':
                dl[1] = length / n_filaments_length  # Along y
            else:
                dl[0] = length / n_filaments_length  # Along x
            
            # Current in filament
            I_fil = I_A * (width / n_filaments_width) * (length / n_filaments_length) / (width * length)
            
            # Vector from source to observation point
            Rx = X - x_fil
            Ry = Y - y_fil
            Rz = Z - z_fil
            R = np.sqrt(Rx**2 + Ry**2 + Rz**2)
            
            # Biot-Savart law: dB = (μ₀/4π) * I * dl × R / R³
            dBx = (MU_0 / (4 * np.pi)) * I_fil * (dl[1]*Rz - dl[2]*Ry) / R**3
            dBy = (MU_0 / (4 * np.pi)) * I_fil * (dl[2]*Rx - dl[0]*Rz) / R**3
            dBz = (MU_0 / (4 * np.pi)) * I_fil * (dl[0]*Ry - dl[1]*Rx) / R**3
            
            # Add contribution
            Bx += dBx
            By += dBy
            Bz += dBz
    
    return Bx, By, Bz

def simulate_wire_field(wire_params, current_params, X_m, Y_m, Z_m):
    """
    Simulate magnetic field for both current polarities.
    
    Returns:
    B_pos_nT, B_neg_nT: B-field arrays [nT] shape (3, grid_size, grid_size)
    """
    
    # Convert currents to Amperes
    I_pos = current_params['I_pos_mA'] * 1e-3
    I_neg = current_params['I_neg_mA'] * 1e-3
    
    # Calculate B fields
    Bx_pos, By_pos, Bz_pos = biot_savart_wire(X_m, Y_m, Z_m, wire_params, I_pos)
    Bx_neg, By_neg, Bz_neg = biot_savart_wire(X_m, Y_m, Z_m, wire_params, I_neg)
    
    # Convert to nT
    B_pos_nT = np.array([Bx_pos, By_pos, Bz_pos]) * 1e9
    B_neg_nT = np.array([Bx_neg, By_neg, Bz_neg]) * 1e9
    
    return B_pos_nT, B_neg_nT

print("Biot-Savart implementation ready.")

## 4. Run Simulation

Calculate the magnetic field maps for both current polarities.

In [ ]:
# Run simulation
print("Running wire field simulation...")
B_sim_pos_nT, B_sim_neg_nT = simulate_wire_field(wire_params, current_params, X_m, Y_m, Z_m)
print("Simulation completed.")

# Calculate magnitudes
B_sim_pos_mag = np.sqrt(np.sum(B_sim_pos_nT**2, axis=0))
B_sim_neg_mag = np.sqrt(np.sum(B_sim_neg_nT**2, axis=0))

print(f"Positive current |B| range: {B_sim_pos_mag.min():.1f} - {B_sim_pos_mag.max():.1f} nT")
print(f"Negative current |B| range: {B_sim_neg_mag.min():.1f} - {B_sim_neg_mag.max():.1f} nT")

## 5. Plot Simulated B-field Maps

Visualize the simulated magnetic field components and magnitude.

In [ ]:
# Plot simulated fields
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)

components = ['Bx', 'By', 'Bz', '|B|']
titles_pos = [f'{comp} (+{current_params["I_pos_mA"]} mA)' for comp in components]
titles_neg = [f'{comp} ({current_params["I_neg_mA"]} mA)' for comp in components]

# Positive current
for i, (comp, title) in enumerate(zip(components, titles_pos)):
    if comp == '|B|':
        data = B_sim_pos_mag
    else:
        comp_idx = ['Bx', 'By', 'Bz'].index(comp)
        data = B_sim_pos_nT[comp_idx]
    
    im = axes[0, i].imshow(data, extent=extent_um, origin='lower', 
                          cmap=CMAP_DIVERGING if comp != '|B|' else CMAP_MAGNITUDE)
    axes[0, i].set_title(title)
    axes[0, i].set_xlabel('x [µm]')
    axes[0, i].set_ylabel('y [µm]')
    plt.colorbar(im, ax=axes[0, i], fraction=0.046, pad=0.04)

# Negative current  
for i, (comp, title) in enumerate(zip(components, titles_neg)):
    if comp == '|B|':
        data = B_sim_neg_mag
    else:
        comp_idx = ['Bx', 'By', 'Bz'].index(comp)
        data = B_sim_neg_nT[comp_idx]
    
    im = axes[1, i].imshow(data, extent=extent_um, origin='lower',
                          cmap=CMAP_DIVERGING if comp != '|B|' else CMAP_MAGNITUDE)
    axes[1, i].set_title(title)
    axes[1, i].set_xlabel('x [µm]')
    axes[1, i].set_ylabel('y [µm]')
    plt.colorbar(im, ax=axes[1, i], fraction=0.046, pad=0.04)

plt.suptitle('Simulated Wire Magnetic Field')
plt.show()

## 6. Compare with Experimental Data

Load and compare the simulated fields with experimental measurements.

In [ ]:
# Reshape experimental data to match simulation grid
def reshape_exp_data(b_df, phase, component, grid_size):
    """Reshape experimental B-field data to 2D grid."""
    phase_df = b_df[(b_df['phase'] == phase) & (b_df['component'] == component)]
    data_2d = np.zeros((grid_size, grid_size))
    
    for _, row in phase_df.iterrows():
        i, j = int(row['row']), int(row['col'])
        data_2d[i, j] = row['value_nT']
    
    return data_2d

# Get experimental data
B_exp_pos_Bx = reshape_exp_data(b_df, 'pos-off', 'Bx', grid_size)
B_exp_pos_By = reshape_exp_data(b_df, 'pos-off', 'By', grid_size)
B_exp_pos_Bz = reshape_exp_data(b_df, 'pos-off', 'Bz', grid_size)
B_exp_neg_Bx = reshape_exp_data(b_df, 'neg-off', 'Bx', grid_size)
B_exp_neg_By = reshape_exp_data(b_df, 'neg-off', 'By', grid_size)
B_exp_neg_Bz = reshape_exp_data(b_df, 'neg-off', 'Bz', grid_size)

B_exp_pos_nT = np.array([B_exp_pos_Bx, B_exp_pos_By, B_exp_pos_Bz])
B_exp_neg_nT = np.array([B_exp_neg_Bx, B_exp_neg_By, B_exp_neg_Bz])

B_exp_pos_mag = np.sqrt(np.sum(B_exp_pos_nT**2, axis=0))
B_exp_neg_mag = np.sqrt(np.sum(B_exp_neg_nT**2, axis=0))

print("Experimental data loaded and reshaped.")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(3, 4, figsize=(16, 12), constrained_layout=True)

components = ['Bx', 'By', 'Bz', '|B|']

for i, comp in enumerate(components):
    # Positive current
    if comp == '|B|':
        sim_data = B_sim_pos_mag
        exp_data = B_exp_pos_mag
    else:
        comp_idx = ['Bx', 'By', 'Bz'].index(comp)
        sim_data = B_sim_pos_nT[comp_idx]
        exp_data = B_exp_pos_nT[comp_idx]
    
    # Simulation
    vmin = min(sim_data.min(), exp_data.min())
    vmax = max(sim_data.max(), exp_data.max())
    
    im1 = axes[0, i].imshow(sim_data, extent=extent_um, origin='lower', 
                           cmap=CMAP_DIVERGING if comp != '|B|' else CMAP_MAGNITUDE,
                           vmin=vmin, vmax=vmax)
    axes[0, i].set_title(f'Sim: {comp} (+{current_params["I_pos_mA"]} mA)')
    plt.colorbar(im1, ax=axes[0, i], fraction=0.046, pad=0.04)
    
    # Experiment
    im2 = axes[1, i].imshow(exp_data, extent=extent_um, origin='lower',
                           cmap=CMAP_DIVERGING if comp != '|B|' else CMAP_MAGNITUDE,
                           vmin=vmin, vmax=vmax)
    axes[1, i].set_title(f'Exp: {comp} (pos-off)')
    plt.colorbar(im2, ax=axes[1, i], fraction=0.046, pad=0.04)
    
    # Difference
    diff = sim_data - exp_data
    im3 = axes[2, i].imshow(diff, extent=extent_um, origin='lower', cmap=CMAP_DIVERGING)
    axes[2, i].set_title(f'Difference: {comp}')
    plt.colorbar(im3, ax=axes[2, i], fraction=0.046, pad=0.04)

for ax in axes.flat:
    ax.set_xlabel('x [µm]')
    ax.set_ylabel('y [µm]')

plt.suptitle('Simulation vs Experiment Comparison (Positive Current)')
plt.show()

In [ ]:
# Quantitative comparison
def calculate_metrics(sim, exp):
    """Calculate comparison metrics."""
    mse = np.mean((sim - exp)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(sim - exp))
    max_diff = np.max(np.abs(sim - exp))
    corr = np.corrcoef(sim.flatten(), exp.flatten())[0, 1]
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'Max_Diff': max_diff,
        'Correlation': corr
    }

print("Quantitative comparison metrics:")
print("\nPositive current:")
for comp in ['Bx', 'By', 'Bz', '|B|']:
    if comp == '|B|':
        metrics = calculate_metrics(B_sim_pos_mag, B_exp_pos_mag)
    else:
        idx = ['Bx', 'By', 'Bz'].index(comp)
        metrics = calculate_metrics(B_sim_pos_nT[idx], B_exp_pos_nT[idx])
    
    print(f"{comp}: RMSE={metrics['RMSE']:.1f} nT, Corr={metrics['Correlation']:.3f}")

print("\nNegative current:")
for comp in ['Bx', 'By', 'Bz', '|B|']:
    if comp == '|B|':
        metrics = calculate_metrics(B_sim_neg_mag, B_exp_neg_mag)
    else:
        idx = ['Bx', 'By', 'Bz'].index(comp)
        metrics = calculate_metrics(B_sim_neg_nT[idx], B_exp_neg_nT[idx])
    
    print(f"{comp}: RMSE={metrics['RMSE']:.1f} nT, Corr={metrics['Correlation']:.3f}")

## 7. Parameter Fitting (Optional)

Implement parameter fitting to optimize wire position and current to match experimental data.

In [ ]:
from scipy.optimize import minimize

def objective_function(params, exp_data, X_m, Y_m, Z_m):
    """
    Objective function for parameter fitting.
    params: [center_x_um, center_y_um, I_pos_mA, I_neg_mA]
    """
    # Update parameters
    fit_params = wire_params.copy()
    fit_params['center_x_um'] = params[0]
    fit_params['center_y_um'] = params[1]
    
    fit_current = current_params.copy()
    fit_current['I_pos_mA'] = params[2]
    fit_current['I_neg_mA'] = params[3]
    
    # Simulate
    B_sim_pos, B_sim_neg = simulate_wire_field(fit_params, fit_current, X_m, Y_m, Z_m)
    
    # Calculate error (RMSE for Bz component, which is dominant)
    error_pos = np.sqrt(np.mean((B_sim_pos[2] - exp_data[0])**2))  # Bz pos
    error_neg = np.sqrt(np.mean((B_sim_neg[2] - exp_data[1])**2))  # Bz neg
    
    return (error_pos + error_neg) / 2

# Initial guess
initial_params = [
    wire_params['center_x_um'], 
    wire_params['center_y_um'],
    current_params['I_pos_mA'],
    current_params['I_neg_mA']
]

# Experimental target data (Bz components)
exp_target = [B_exp_pos_nT[2], B_exp_neg_nT[2]]

print("Fitting wire parameters...")
print(f"Initial parameters: {initial_params}")

# Note: This is computationally expensive, uncomment to run
# result = minimize(objective_function, initial_params, args=(exp_target, X_m, Y_m, Z_m),
#                  method='Nelder-Mead', options={'maxiter': 50})

# print(f"Optimized parameters: {result.x}")
# print(f"Final error: {result.fun:.3f} nT")

print("Parameter fitting code ready (commented out due to computational cost).")

## 8. Save Simulation Results

Export the simulation results and comparison metrics.

In [ ]:
# Save simulation results
rows = []

for i in range(grid_size):
    for j in range(grid_size):
        # Simulation data
        for phase, B_sim in [('pos', B_sim_pos_nT), ('neg', B_sim_neg_nT)]:
            for comp_idx, comp in enumerate(['Bx', 'By', 'Bz']):
                row_dict = {
                    "record_type": "simulation",
                    "phase": phase,
                    "component": comp,
                    "row": i,
                    "col": j,
                    "x_um": X_um[i, j],
                    "y_um": Y_um[i, j],
                    "value_nT": B_sim[comp_idx, i, j],
                    "unit": "nT"
                }
                rows.append(row_dict)
        
        # Experimental data for comparison
        for phase, B_exp in [('pos-off', B_exp_pos_nT), ('neg-off', B_exp_neg_nT)]:
            for comp_idx, comp in enumerate(['Bx', 'By', 'Bz']):
                row_dict = {
                    "record_type": "experiment",
                    "phase": phase,
                    "component": comp,
                    "row": i,
                    "col": j,
                    "x_um": X_um[i, j],
                    "y_um": Y_um[i, j],
                    "value_nT": B_exp[comp_idx, i, j],
                    "unit": "nT"
                }
                rows.append(row_dict)

# Add simulation metadata
metadata_rows = [
    {"record_type": "metadata", "phase": "", "component": "", "row": "", "col": "", "x_um": "", "y_um": "", "value_nT": "wire_center_x_um", "unit": wire_params['center_x_um']},
    {"record_type": "metadata", "phase": "", "component": "", "row": "", "col": "", "x_um": "", "y_um": "", "value_nT": "wire_center_y_um", "unit": wire_params['center_y_um']},
    {"record_type": "metadata", "phase": "", "component": "", "row": "", "col": "", "x_um": "", "y_um": "", "value_nT": "wire_length_um", "unit": wire_params['length_um']},
    {"record_type": "metadata", "phase": "", "component": "", "row": "", "col": "", "x_um": "", "y_um": "", "value_nT": "wire_width_um", "unit": wire_params['width_um']},
    {"record_type": "metadata", "phase": "", "component": "", "row": "", "col": "", "x_um": "", "y_um": "", "value_nT": "wire_height_um", "unit": wire_params['height_um']},
    {"record_type": "metadata", "phase": "", "component": "", "row": "", "col": "", "x_um": "", "y_um": "", "value_nT": "I_pos_mA", "unit": current_params['I_pos_mA']},
    {"record_type": "metadata", "phase": "", "component": "", "row": "", "col": "", "x_um": "", "y_um": "", "value_nT": "I_neg_mA", "unit": current_params['I_neg_mA']},
]

rows.extend(metadata_rows)

sim_results_df = pd.DataFrame(rows)
sim_results_df.to_csv(SIMULATION_RESULTS_CSV, index=False)

print(f"Saved simulation results to {SIMULATION_RESULTS_CSV}")
print(f"Total records: {len(sim_results_df)}")
print(f"Simulation records: {len(sim_results_df[sim_results_df['record_type'] == 'simulation'])}")
print(f"Experiment records: {len(sim_results_df[sim_results_df['record_type'] == 'experiment'])}")
print(f"Metadata records: {len(sim_results_df[sim_results_df['record_type'] == 'metadata'])}")

## Summary

This notebook provides a complete workflow for:
1. Loading experimental NV sensing B-field data
2. Simulating magnetic fields from current-carrying wires
3. Comparing simulation and experiment
4. Quantitative analysis and parameter fitting
5. Exporting results for further analysis

The simulation uses the Biot-Savart law with finite-width wire discretization for accurate modeling of real experimental conditions.